# 04 — Deep Learning

Three feed-forward architectures. No CNN/LSTM/transformer: tabular data has no spatial or temporal structure.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

import pandas as pd, numpy as np
import config
from src import data_loader as dl

from src import train_dl, utils
from src.preprocessing import build_preprocessor
utils.set_seeds()

In [ ]:
df = dl.load_raw()
splits = dl.make_splits(df)
print(splits.sizes)

## Preprocess

Neural networks need scaled inputs. The preprocessor is fitted on train only.

In [ ]:
pre = build_preprocessor(splits.X_train, scale=True)
Xtr = pre.fit_transform(splits.X_train).astype('float32')
Xva = pre.transform(splits.X_val).astype('float32')
ytr = splits.y_train.to_numpy().astype('float32')
yva = splits.y_val.to_numpy().astype('float32')
print('encoded features:', Xtr.shape[1])

## Architectures

Layer widths derive from the actual encoded feature count.

In [ ]:
for name, (builder, note) in train_dl.DL_ARCHITECTURES.items():
    m = builder(Xtr.shape[1])
    print('%s: %s parameters — %s' % (name, format(m.count_params(), ','), note))

In [ ]:
model = train_dl.build_deep_mlp(Xtr.shape[1])
model.summary()

## Train one network

Early stopping monitors the validation split — never the test set.

In [ ]:
import keras
model.compile(optimizer=keras.optimizers.Adam(config.LEARNING_RATE),
              loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(Xtr, ytr, validation_data=(Xva, yva),
                    epochs=20,
                    batch_size=config.BATCH_SIZE,
                    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss',
                               patience=config.EARLY_STOPPING_PATIENCE,
                               restore_best_weights=True)],
                    verbose=0)
print('epochs run:', len(history.history['loss']))

## Training curves

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['accuracy'], label='train')
ax[0].plot(history.history['val_accuracy'], label='validation')
ax[0].set_title('Accuracy vs epochs'); ax[0].legend()
ax[1].plot(history.history['loss'], label='train')
ax[1].plot(history.history['val_loss'], label='validation')
ax[1].set_title('Loss vs epochs'); ax[1].legend()
plt.show()

## Compare against the saved results

In [ ]:
if config.DL_RESULTS_CSV.exists():
    display(pd.read_csv(config.DL_RESULTS_CSV).round(4))
else:
    print('Run `python train.py --mode full` first.')